In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import plotly.express as px



In [ ]:

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Data Analysis/Data sets/pizza_sales.csv")
df.head()

In [ ]:
df.dtypes

In [ ]:
df.tail()

In [ ]:
print("The MetaDate of the Dataset: ",df.shape)

In [ ]:
#print("The Rows of the Dataset: ",df.shape(0))
#print("The Columns of the Dataset: ",df.shape(1))

In [ ]:
df.info()

In [ ]:
df.describe()

**KPI's**

In [ ]:
total_revenue = df["total_price"].sum()
total_pizzas_sold = df["quantity"].sum()
total_orders = df["order_id"].nunique()
avg_order_value = total_revenue / total_orders
avg_pizza_per_order = total_pizzas_sold / total_orders

print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Pizzas Sold: {total_pizzas_sold}")
print(f"Total Orders: {total_orders}")
print(f"Average Order Value: ${avg_order_value:,.2f}")
print(f"Average Pizzas per Order: {avg_pizza_per_order:.2f}")


**Ingredient Analysis**

In [ ]:
Ingredient = (
    df['pizza_ingredients']
    .str.split(',')
    .explode()
    .str.strip()
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'Ingredients', 0: 'Count'})
)

print(Ingredient.head(10))


In [ ]:
plt.figure(figsize=(8,4))

if 'order_date' not in df.columns:
    df = df.reset_index()

df['order_date'] = pd.to_datetime(df['order_date'], format='%d-%m-%Y', errors='coerce')
df = df.dropna(subset=['order_date'])

df['day_of_week'] = df['order_date'].dt.day_name()


orders_by_day = (
    df.groupby('day_of_week')
    .agg(total_orders=('order_id', 'nunique'))
    .reindex(['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
    .reset_index()
)

bars = plt.bar(orders_by_day['day_of_week'], orders_by_day['total_orders'], color='#88B04B')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 5, int(yval), ha='center', va='bottom')

plt.title('Pizza Orders Distribution by Day of the Week')
plt.xlabel('Day of Week')
plt.ylabel('Total Orders')
plt.tight_layout()
plt.show()

Friday is the busiest day with 3,538 orders, representing peak demand. Sunday is the slowest day with 2,624 orders, indicating lower customer activity.Orders gradually increase midweek and peak towards the end of the week, then decline on the weekend (Sunday).

In [ ]:
if 'order_date' not in df.columns:
    df = df.reset_index()

df['order_date'] = pd.to_datetime(df['order_date'], format='%d-%m-%Y', errors='coerce')
df = df.dropna(subset=['order_date'])


df['day_of_week'] = df['order_date'].dt.day_name()

revenue_by_day = (
    df.groupby('day_of_week')['total_price']
    .sum()
    .reindex(['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
)


plt.figure(figsize=(8,4))
bars = plt.bar(revenue_by_day.index, revenue_by_day.values, color='#6B5B95')


for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 50, f"${yval:,.0f}", ha='center', va='bottom')

plt.title('Total Revenue by Day of the Week')
plt.xlabel('Day of Week')
plt.ylabel('Revenue ($)')
plt.tight_layout()


plt.savefig("total_revenue_by_day.png", dpi=300)
plt.show()


In [ ]:
if 'order_time' not in df.columns:
    df = df.reset_index()

df['order_time'] = pd.to_datetime(df['order_time'], format='%H:%M:%S', errors='coerce')
df = df.dropna(subset=['order_time'])

df['order_hour'] = df['order_time'].dt.hour


orders_by_hour = df.groupby('order_hour')['order_id'].nunique()


plt.figure(figsize=(10,5))
bars = plt.bar(orders_by_hour.index, orders_by_hour.values, color='#FF6F61')

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 2, int(yval), ha='center', va='bottom')

plt.title('Total Orders by Hour of the Day')
plt.xlabel('Hour of Day')
plt.ylabel('Total Orders')
plt.xticks(range(0,24))
plt.tight_layout()


plt.savefig("total_orders_by_hour.png", dpi=300)
plt.show()


In [ ]:
if 'order_date' not in df.columns:
    df = df.reset_index()

df['order_date'] = pd.to_datetime(df['order_date'], format='%d-%m-%Y', errors='coerce')
df = df.dropna(subset=['order_date'])
df['month_name'] = df['order_date'].dt.strftime('%B')

orders_by_month = df.groupby('month_name')['order_id'].nunique()
months_order = ['January','February','March','April','May','June',
                'July','August','September','October','November','December']
orders_by_month = orders_by_month.reindex(months_order)

plt.figure(figsize=(10,5))
plt.fill_between(orders_by_month.index, orders_by_month.values, color='lightcoral', alpha=0.6)
plt.plot(orders_by_month.index, orders_by_month.values, marker='o', color='darkred', linewidth=2)

for i, value in enumerate(orders_by_month.values):
    plt.text(i, value + 2, int(value), ha='center', va='bottom')

plt.title('Total Orders by Month')
plt.xlabel('Month')
plt.ylabel('Total Orders')
plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.savefig("total_orders_by_month.png", dpi=300)
plt.show()


In [ ]:
category_sales = df.groupby('pizza_category')['total_price'].sum()
category_percentage = (category_sales / category_sales.sum()) * 100

plt.figure(figsize=(5,5))
wedges, texts, autotexts = plt.pie(
    category_percentage,
    labels=category_percentage.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=['skyblue','lightgreen','salmon'],
    wedgeprops=dict(width=0.5)
)
plt.title('Percentage of Sales by Pizza Category')
plt.tight_layout()

plt.savefig("Percentage of Sales by Pizza Category.png", dpi=300)
plt.show()


In [ ]:
size_category_sales = df.groupby(['pizza_size', 'pizza_category'])['total_price'].sum().unstack()
size_category_percentage = size_category_sales.div(size_category_sales.sum().sum()) * 100

plt.figure(figsize=(8,6))
sns.heatmap(
    size_category_percentage,
    annot=True,
    fmt=".1f",
    cmap='YlOrRd',
    cbar_kws={'label':'Percentage (%)'}
)
plt.title('Percentage of Sales by Pizza Size and Category')
plt.ylabel('Pizza Size')
plt.xlabel('Pizza Category')
plt.tight_layout()


plt.savefig("Percentage of Sales by Pizza Size and Category.png", dpi=300)
plt.show()


In [ ]:
pizza_sold_category = df.groupby('pizza_category')['quantity'].sum()

plt.figure(figsize=(8,5))
bars = plt.bar(pizza_sold_category.index, pizza_sold_category.values, color=['#FF6F61','#6B5B95','#88B04B'])

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 2, int(yval), ha='center', va='bottom')

plt.title('Total Pizzas Sold by Pizza Category')
plt.xlabel('Pizza Category')
plt.ylabel('Total Pizzas Sold')
plt.tight_layout()


plt.savefig("total_pizzas_sold_by_category.png", dpi=300)
plt.show()


In [ ]:
top5_pizza = df.groupby('pizza_name')['quantity'].sum().sort_values(ascending=False).head(5)

plt.figure(figsize=(8,5))
bars = plt.bar(top5_pizza.index, top5_pizza.values, color=['#FF6F61','#6B5B95','#88B04B','#F7DC6F','#5DADE2'])

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 2, int(yval), ha='center', va='bottom')

plt.title('Top 5 Best Selling Pizzas')
plt.xlabel('Pizza Name')
plt.ylabel('Quantity Sold')
plt.xticks(rotation=45)
plt.tight_layout()


plt.savefig("top5_best_selling_pizzas.png", dpi=300)
plt.show()


In [ ]:
bottom5_pizza = df.groupby('pizza_name')['quantity'].sum().sort_values(ascending=True).head(5)

plt.figure(figsize=(8,5))
bars = plt.bar(bottom5_pizza.index, bottom5_pizza.values, color=['#FF6F61','#6B5B95','#88B04B','#F7DC6F','#5DADE2'])

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 1, int(yval), ha='center', va='bottom')

plt.title('Bottom 5 Least Selling Pizzas')
plt.xlabel('Pizza Name')
plt.ylabel('Quantity Sold')
plt.xticks(rotation=45)
plt.tight_layout()


plt.savefig("bottom5_least_selling_pizzas.png", dpi=300)
plt.show()


In [ ]:
df.columns = [c.strip() for c in df.columns]

df['order_datetime'] = pd.to_datetime(df['order_date'].astype(str) + ' ' + df['order_time'].astype(str), errors='coerce')
df['order_hour'] = df['order_datetime'].dt.hour

# Peak Order Times by Category
category_hourly = df.groupby(['pizza_category', 'order_hour']).agg(total_pizzas=('quantity','sum')).reset_index()

plt.figure(figsize=(10,6))
for category in category_hourly['pizza_category'].unique():
    temp = category_hourly[category_hourly['pizza_category'] == category]
    plt.plot(temp['order_hour'], temp['total_pizzas'], marker='o', label=category)

plt.title("Peak Order Times by Pizza Category")
plt.xlabel("Hour of Day")
plt.ylabel("Total Pizzas Sold")
plt.xticks(range(0,24))
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig("peak_order_times_by_category.png", dpi=300)
plt.show()


In [ ]:
# Peak Order Times by Size
size_hourly = df.groupby(['pizza_size', 'order_hour']).agg(total_pizzas=('quantity','sum')).reset_index()

plt.figure(figsize=(10,6))
for size in size_hourly['pizza_size'].unique():
    temp = size_hourly[size_hourly['pizza_size'] == size]
    plt.plot(temp['order_hour'], temp['total_pizzas'], marker='o', label=size)

plt.title("Peak Order Times by Pizza Size")
plt.xlabel("Hour of Day")
plt.ylabel("Total Pizzas Sold")
plt.xticks(range(0,24))
plt.grid(True)
plt.legend()
plt.tight_layout()


plt.savefig("peak_order_times_by_size.png", dpi=300)
plt.show()


In [ ]:
import os

# Ingredient Popularity
df['ingredients_list'] = df['pizza_ingredients'].fillna('').astype(str).apply(lambda s: [i.strip().lower() for i in s.split(',') if i.strip()])
ingredients_exploded = df.explode('ingredients_list')

ingredient_usage = ingredients_exploded.groupby('ingredients_list').agg(
    ingredient_uses=('ingredients_list','count'),
    pizzas_using=('pizza_name','nunique')
).sort_values('ingredient_uses', ascending=False)


# Plot top 20 ingredients
plt.figure(figsize=(12,6))
ingredient_usage.head(20)['ingredient_uses'].plot(kind='bar')
plt.title("Top 20 Ingredients by Usage")
plt.ylabel("Total Uses")
plt.tight_layout()


plt.savefig("top20_ingredients_usage.png", dpi=300)
plt.show()


In [ ]:
import os

if 'cost' not in df.columns:
    df['cost'] = df['unit_price'] * 0.6

df['profit'] = df['total_price'] - (df['cost'] * df['quantity'])

profit_stats = df.groupby('pizza_name').agg(
    total_revenue=('total_price','sum'),
    total_cost=('cost','sum'),
    total_profit=('profit','sum'),
    total_quantity=('quantity','sum')
).sort_values('total_profit', ascending=False)


plt.figure(figsize=(10,6))
profit_stats.head(10)['total_profit'].plot(kind='bar', color='green')
plt.title("Top 10 Profitable Pizzas")
plt.ylabel("Profit")
plt.tight_layout()


plt.savefig("top10_profitable_pizzas.png", dpi=300)
plt.show()


In [ ]:
recommendations = []

# Promote high-profit pizzas
top_profitable = profit_stats.head(5).index.tolist()
recommendations.append(f"Promote top profitable pizzas: {', '.join(top_profitable)}")

# Consider removing low-selling/low-profit pizzas
bottom_profit = profit_stats.tail(5)
low_sales_pizzas = bottom_profit[bottom_profit['total_profit'] <= 0].index.tolist()
if low_sales_pizzas:
    recommendations.append(f"Consider removing or revising low-profit pizzas: {', '.join(low_sales_pizzas)}")

# Peak time promotions
peak_hours_category = category_hourly.groupby('pizza_category')['total_pizzas'].idxmax().apply(lambda x: category_hourly.loc[x, 'order_hour'])
for cat, hour in peak_hours_category.items():
    recommendations.append(f"Run promotions for {cat} pizzas around peak hour {hour}:00")



print("Recommendations generated:")
for r in recommendations:
    print("-", r)

In [ ]:
df.columns = [c.strip() for c in df.columns]
df['order_datetime'] = pd.to_datetime(df['order_date'].astype(str) + ' ' + df['order_time'].astype(str), errors='coerce')
df['month'] = df['order_datetime'].dt.month
df['year'] = df['order_datetime'].dt.year

#total orders abd revenue by month
monthly_sales = df.groupby(['year','month']).agg(
    total_orders=('order_id','nunique'),
    total_pizzas=('quantity','sum'),
    total_revenue=('total_price','sum')
).reset_index()

monthly_sales['month_year'] = monthly_sales['year'].astype(str) + "-" + monthly_sales['month'].astype(str).str.zfill(2)

# Plot Total Revenue by Month
plt.figure(figsize=(12,6))
plt.plot(monthly_sales['month_year'], monthly_sales['total_revenue'], marker='o', color='blue')
plt.xticks(rotation=45)
plt.xlabel("Month-Year")
plt.ylabel("Total Revenue")
plt.title("Monthly Revenue Trend")
plt.grid(True)
plt.tight_layout()
plt.savefig("monthly_revenue_trend.png", dpi=300)
plt.show()


In [ ]:
# Plot Total Orders by Month
plt.figure(figsize=(12,6))
plt.plot(monthly_sales['month_year'], monthly_sales['total_orders'], marker='o', color='green')
plt.xticks(rotation=45)
plt.xlabel("Month-Year")
plt.ylabel("Total Orders")
plt.title("Monthly Orders Trend")
plt.grid(True)
plt.tight_layout()
plt.savefig("monthly_orders_trend.png", dpi=300)
plt.show()


In [ ]:
seasonal_trends = df.groupby(df['order_datetime'].dt.month).agg(
    avg_orders=('order_id','nunique'),
    avg_revenue=('total_price','sum')
).sort_index()
seasonal_trends.index = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

plt.figure(figsize=(12,6))
plt.plot(seasonal_trends.index, seasonal_trends['avg_revenue'], marker='o', color='red')
plt.xlabel("Month")
plt.ylabel("Average Revenue")
plt.title("Average Monthly Revenue (Seasonal Trend)")
plt.grid(True)
plt.tight_layout()


plt.savefig("seasonal_revenue_trend.png", dpi=300)
plt.show()


**Which pizzas generate the most revenue**

In [ ]:


# Top 5 revenue-generating pizzas
top5_revenue = (
    df.groupby('pizza_name')['total_price']
    .sum()
    .sort_values(ascending=False)
    .head(5)
)

plt.figure(figsize=(8,5))
bars = plt.bar(
    top5_revenue.index,
    top5_revenue.values,
    color=['#FF6F61','#6B5B95','#88B04B','#F7DC6F','#5DADE2']
)

for bar in bars:
    yval = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        yval,
        f"{yval:.2f}",
        ha='center',
        va='bottom'
    )

plt.title('Top 5 Pizzas by Revenue')
plt.xlabel('Pizza Name')
plt.ylabel('Total Revenue')
plt.xticks(rotation=45)
plt.tight_layout()


plt.savefig("top_5_pizzas_by_revenue.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:


# Bottom 5 revenue-generating pizzas
least5_revenue = (
    df.groupby('pizza_name')['total_price']
    .sum()
    .sort_values()
    .head(5)
)

plt.figure(figsize=(8,5))
bars = plt.bar(
    least5_revenue.index,
    least5_revenue.values,
    color=['#D98880','#F5B7B1','#E6B0AA','#CD6155','#922B21']
)

# Add value labels on bars
for bar in bars:
    yval = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        yval,
        f"{yval:.2f}",
        ha='center',
        va='bottom'
    )

plt.title('Least 5 Pizzas by Revenue')
plt.xlabel('Pizza Name')
plt.ylabel('Total Revenue')
plt.xticks(rotation=45)
plt.tight_layout()


plt.savefig("least_5_pizzas_by_revenue.png", dpi=300, bbox_inches='tight')
plt.show()
